# Bài Thực Hành Part 1: Giải Hệ Phương Trình Tuyến Tính

Notebook này minh họa các phương pháp giải hệ phương trình tuyến Tính (AX = b) bao gồm:
- **Phương pháp khử Gauss** (Gaussian Elimination)
- **Tính định thức** (Determinant)
- **Tìm ma trận nghịch đảo** (Matrix Inverse)
- **Tính hạng ma trận và cơ sở** (Rank and Basis)

Các test case đã bao gồm:
1. Hệ có nghiệm duy nhất
2. Hệ có vô số nghiệm (phụ thuộc vào tham số)
3. Hệ vô nghiệm (mâu thuẫn)

In [36]:
import numpy as np
from gaussian import gaussian_eliminate
from verify import verify_solution, verify_determinant, verify_inverse, verify_rank 
from determinant import determinant
from inverse import inverse
from rank_basis import rank_and_basis

## 1. Các Hàm Hiển Thị Kết Quả

Các hàm hỗ trợ in kết quả một cách đẹp mắt:

In [26]:
def print_solution(x, use_fraction=False):
    if x is None:
        print("Inconsistent system of equations.\n")
        return
        
    # Thiết lập ngưỡng sai số: nếu dùng Fraction thì eps = 0
    eps = 0 if use_fraction else 1e-12
    
    for i, expr in enumerate(x):
        terms = []
        const_val = expr.get('const', 0)
        
        # Xử lý phần hằng số
        # Nếu dùng Fraction, in trực tiếp. Nếu dùng float, làm tròn
        if abs(const_val) > eps or len(expr) == 1:
            val_str = str(const_val) if use_fraction else str(round(const_val, 4))
            terms.append(val_str)
            
        # Xử lý các biến tham số t_i
        for key, val in expr.items():
            if key != 'const' and abs(val) > eps:
                # Xác định dấu
                sign = "+" if val > 0 else "-"
                
                # Xử lý hệ số
                abs_val = abs(val)
                if abs_val == 1:
                    coeff = "" # Không in 1*t_i, chỉ in t_i
                else:
                    coeff = f"{abs_val}*" if use_fraction else f"{round(abs_val, 4)}*"
                
                # Nếu đã có phần tử trước đó trong terms, thêm dấu cách cho đẹp
                if terms:
                    terms.append(f"{sign} {coeff}{key}")
                else:
                    # Nếu là phần tử đầu dòng, dấu trừ thì giữ, dấu cộng thì bỏ
                    prefix = "" if sign == "+" else "-"
                    terms.append(f"{prefix}{coeff}{key}")
                
        # Ghép chuỗi lại
        expr_str = " ".join(terms).replace(" + -", " - ").replace(" - +", " - ")
        
        # Xử lý trường hợp đầu chuỗi có "+ "
        if expr_str.startswith("+ "): 
            expr_str = expr_str[2:]
            
        print(f"x_{i} = {expr_str}")
    print()

In [27]:
def print_matrix(title, M, use_fraction=False):
    """
    In ma trận với tiêu đề và định dạng đẹp.
    Hỗ trợ cả số thực (float) và phân số (Fraction).
    """
    print("\n{}:".format(title))
    for row in M:
        row_str = ""
        for val in row:
            if use_fraction:
                # Căn lề phải 12 ký tự để các cột thẳng hàng
                row_str += "  {:>12}".format(str(val))
            else:
                # Với float, dùng định dạng số thực
                row_str += "  {:10.4f}".format(val)
        print(row_str)

## 2. Test Case 1: Hệ có Nghiệm Duy Nhất

Hệ phương trình:
```
2x₁ + x₂ - x₃ = 8
-3x₁ - x₂ + 2x₃ = -11
-2x₁ + x₂ + 2x₃ = -3
```

Kỳ vọng: Ma trận hệ số A có định thức khác 0, do đó có ma trận nghịch đảo duy nhất.

In [28]:

A = [[2, 1, -1], 
    [-3, -1, 2], 
    [-2, 1, 2]]
b = [8, -11, -3]

# Khử Gauss
U, x, swaps = gaussian_eliminate(A, b, use_fraction=True)

print("Upper triangular matrix U:")
print_matrix("U", U, use_fraction=True)

print("\nModified right-hand side x:")
print_solution(x, use_fraction=True)

print(f"Number of row swaps: {swaps}\n")

# Kiểm chứng
verify_solution(A, x, b)

Upper triangular matrix U:

U:
            -3            -1             2
             0           5/3           2/3
             0             0           1/5

Modified right-hand side x:
x_0 = 2
x_1 = 3
x_2 = -1

Number of row swaps: 2

=> VERIFICATION SUCCESS: Solution x is correct (A*x == b).


## 3. Test Case 2: Hệ có Vô Số Nghiệm

Hệ phương trình:
```
x₁ + x₂ + x₃ = 1
x₁ + 2x₂ + 3.5x₃ = 2
2x₁ + 3x₂ + 4.5x₃ = 3
```

Kỳ vọng: Hàng thứ 3 phụ thuộc tuyến tính vào hai hàng đầu, do đó hệ có vô số nghiệm phụ thuộc vào tham số tự do.

In [29]:
A_inf = [
    [1.0, 1.0, 1.0],
    [1.0, 2.0, 3.5],
    [2.0, 3.0, 4.5]
]

b_inf = [1.0, 2.0, 3.0]

# Khử Gauss
U, x, swaps = gaussian_eliminate(A_inf, b_inf, True)

print("Upper triangular matrix U:")
print_matrix("U", U, use_fraction=True)

print("\nModified right-hand side x:")
print_solution(x, use_fraction=True)

print(f"Number of row swaps: {swaps}\n")

# Kiểm chứng
verify_solution(A_inf, x, b_inf)

Upper triangular matrix U:

U:
             2             3           9/2
             0           1/2           5/4
             0             0             0

Modified right-hand side x:
x_0 = 3/2*t_2
x_1 = 1 - 5/2*t_2
x_2 = t_2

Number of row swaps: 1

=> VERIFICATION SUCCESS: Solution x is correct (A*x == b).


## 4. Test Case 3: Hệ Vô Nghiệm

Hệ phương trình:
```
x₁ + x₂ + x₃ = 3
x₁ + 2x₂ + 3x₃ = 6
2x₁ + 3x₂ + 4x₃ = 8
```

Kỳ vọng: Hệ không có nghiệm vì các phương trình mâu thuẫn (không tương thích). Điều này xuất hiện khi có hàng 0 ở vế trái nhưng vế phải khác 0.

In [30]:
A_no_sol = [
    [1.0, 1.0, 1.0],
    [1.0, 2.0, 3.0],
    [2.0, 3.0, 4.0]
]

b_no_sol = [3.0, 6.0, 8.0]

# Khử Gauss
U, x, swaps = gaussian_eliminate(A_no_sol, b_no_sol)

print("Upper triangular matrix U:")
print_matrix("U", U, use_fraction=False)

print("\nModified right-hand side x:")
print_solution(x, use_fraction=False)

print(f"Number of row swaps: {swaps}\n")

# Kiểm chứng
verify_solution(A_no_sol, x, b_no_sol)

Upper triangular matrix U:

U:
      2.0000      3.0000      4.0000
      0.0000      0.5000      1.0000
      0.0000      0.0000      0.0000

Modified right-hand side x:
Inconsistent system of equations.

Number of row swaps: 1

NumPy confirms the system is inconsistent (no solution). Skipping allclose verification.
=> VERIFICATION SUCCESS: Your function also correctly identified the system as inconsistent.


## 5. Tính Định Thức (Determinant)

Tính định thức của ma trận từ các test case và kiểm chứng kết quả:

In [31]:
    
# Determinant
det = determinant(A)
# Verify determinant
verify_determinant(A, det)

# Determinant for infinite solution case
det_inf = determinant(A_inf)
verify_determinant(A_inf, det_inf)

Determinant calculated: -1
Determinant of NumPy: -1
=> VERIFICATION SUCCESS: Determinant is correct.
Determinant calculated: 0
Determinant of NumPy: 0
=> VERIFICATION SUCCESS: Determinant is correct.


## 6. Tìm Ma Trận Nghịch Đảo (Matrix Inverse)

Ma trận chỉ có nghịch đảo khi định thức khác 0. Dưới đây tìm nghịch đảo cho các test case phù hợp:

In [32]:
# Inverse
try:
    A_inv = inverse(A)

except ValueError:
    A_inv = None

# Verify inverse
verify_inverse(A, A_inv)
if A_inv is not None:
    print(np.array(A_inv))


# Inverse for infinite solution case
try:
    A_inf_inv = inverse(A_inf)

except ValueError:
    A_inf_inv = None

verify_inverse(A_inf, A_inf_inv)
if A_inf_inv is not None:
    print(np.array(A_inf_inv))   

=> VERIFICATION SUCCESS: Inverse matrix is correct.
[[ 4.  3. -1.]
 [-2. -2.  1.]
 [ 5.  4. -1.]]
=> VERIFICATION SUCCESS: The matrix is singular (not invertible).


## 7. Tính Hạng Ma Trận và Cơ Sở (Rank and Basis)

Hạng (rank) của ma trận là số chiều của không gian hàng hoặc cột. Cơ sở là tập hợp các vector độc lập tuyến tính tạo thành không gian đó.

In [37]:
# Rank and basis
rank, column_basis, row_basis, null_basis = rank_and_basis(A)
verify_rank(A, rank)


# Rank and basis for infinite solution case
rank_inf, column_basis_inf, row_basis_inf, null_basis_inf = rank_and_basis(A_inf)
verify_rank(A_inf, rank_inf)

Rank tự tính: 3
Rank của NumPy: 3
=> VERIFICATION SUCCESS: Rank is correct.
Rank tự tính: 2
Rank của NumPy: 2
=> VERIFICATION SUCCESS: Rank is correct.
